# BONITA — Italian task generator for boolean QA (Minerva-7B-instruct)

*Francesco Baiocchi, Leonardo Petrilli — Task 3, course project work*

## Goal

BONITA (from the [Bonito paper](https://arxiv.org/pdf/2402.18334), *"Learning to Generate Instruction Tuning Datasets for Zero-Shot Task Adaptation"*) is, in its general form, a pipeline that takes **raw, unannotated text** and produces an **instruction-tuning dataset for an arbitrary task**, via a meta-template that tells a language model what kind of task to synthesize from a given passage (Figure 2 of the paper). Our brief for Task 3 was to instantiate this idea for Italian, on top of the datasets in the [ita-bench collection](https://huggingface.co/collections/sapienzanlp/ita-bench-italian-benchmarks-for-llms-66337ca59e6df7d7d4933896), by fine-tuning an Italian instruction-tuned model to act as the task generator.

---

Given our timeline and the compute we had access to (DISI HPC cluster, one L40 48GB GPU), we deliberately scoped down to a **single task type** first rather than attempting the full multi-task pipeline from the start: **boolean question answering**, using `sapienzanlp/boolq_italian` as the source of training examples. We picked it because it is one of the few ita-bench datasets where passage, question, and label are already cleanly separated.

Concretely, this means: we take an **already-annotated** benchmark and turn each example into a BONITA meta-template pair — **input** = `<|tasktype|>` + the passage, **output** = an instruction, a question about the passage, and `<|pipe|>` followed by the answer (`Risposta: Vero` / `Risposta: Falso`). We then **QLoRA fine-tune** `sapienzanlp/Minerva-7B-instruct-v1.0` on these pairs, so that given a new passage it **generates a boolean QA task** (question + answer), i.e. one instruction-tuning example.

Our evaluation is **intrinsic**: for each held-out passage, we let the model generate a task and compare the generated question with **all** the reference questions that the benchmark [boolq_italian](https://huggingface.co/datasets/sapienzanlp/boolq_italian) pairs with that passage, using BLEU, METEOR and BERTScore. Since a passage admits many valid questions (the task is one-to-many), we also report format adherence and answer-level statistics, and we compare against the base model on the same prompts.

## Table of contents

1. Environment setup
2. Dataset & meta-template prompt construction
3. Train / dev / test split by passage
4. Base model setup
5. Training
6. Loading the trained adapter for inference
7. Evaluation of the generated tasks: automatic metrics
8. Base vs. fine-tuned comparison

## 1. Environment setup

In [ ]:
# Run once per environment. Safe to skip if the environment already has these installed
# (e.g. via `uv sync` from this project's pyproject.toml / uv.lock).

!pip install -q trl transformers peft bitsandbytes datasets jinja2 pandas
!pip install -q evaluate nltk bert_score

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc
import json
import random
import re
from collections import defaultdict

import evaluate
import nltk
import pandas as pd
import torch
import transformers
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from jinja2 import Template
from tqdm.notebook import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)

nltk.download("wordnet")


def fix_random(seed: int) -> None:
    """
    Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


SEED = 42
fix_random(seed=SEED)

## 2. Dataset & meta-template prompt construction

We use `sapienzanlp/boolq_italian` train+validation splits merged because each example already separates passage, question, and boolean label cleanly. For context: other ita-bench datasets, e.g. `arc_italian`, don't have this separation. The merged pool is then re-split **by passage** in Section 3.

Each example is converted to a `prompt` / `completion` pair using BONITA's meta-template:

- **`prompt`** (model input): `<|tasktype|>` + `boolean question answering` + `<|context|>` + the passage + `<|task|>`;

- **`completion`** (what the model learns to generate): an instruction (one of two variants, picked at random), the question followed by *"Vero o Falso?"*, then `<|pipe|>` and the answer `Risposta: Vero` / `Risposta: Falso`.

So the target the model learns is the whole **task** (instruction + question + answer), not just the label.

> **Fix w.r.t. the original notebook:** the output template used to start with a `{{ context }}` placeholder that was never filled (Jinja renders it as an empty string), and `"{{ context }}"` was also registered as a special token that never appeared in the data. Both are removed; the rendered completions are unchanged (they still start with a newline).

> **Note:** A personal Hugging Face access token is required, please authenticate with Hugging Face before executing the following cells.

In [2]:
dataset = load_dataset("sapienzanlp/boolq_italian")

In [3]:
ds = concatenate_datasets([dataset["train"], dataset["validation"]])

In [4]:
ids = []
questions = []
contexts = []
labels = []
for i in range(len(ds)):
    ids.append(ds[i]["id"])
    questions.append(ds[i]["input_translation"] + " Vero o Falso?")
    contexts.append(ds[i]["metadata"]["passage_translation"])
    if ds[i]["label"]:
        labels.append("Risposta: Vero")
    else:
        labels.append("Risposta: Falso")

In [5]:
INSTRUCTION_VARIANTS = [
    "Avendo letto il precedente passaggio, rispondi alla seguente domanda:",
    "Letto il precedente contesto, rispondi alla domanda:",
]


def generate_QA_dataset(contexts: list[str], questions: list[str], labels: list[str]) -> Dataset:
    """
    Uses Jinja2 templates to format prompts and completions

    Randomly picks one of two instruction variants

    Returns a Hugging Face Dataset with prompt and completion fields
    """
    input_template = (
        "<|tasktype|>\nboolean question answering\n<|context|>\n{{ passage }}\n<|task|>"
    )
    output_template = "\n{{ instruction }} {{ question }}\n<|pipe|>\n{{ answer }}"

    input_template_jinja = Template(input_template)
    output_template_jinja = Template(output_template)

    dataset = {"prompt": [], "completion": []}
    for i in range(len(contexts)):
        data_input = {"passage": contexts[i]}
        data_output = {
            "instruction": random.choice(INSTRUCTION_VARIANTS),
            "question": questions[i],
            "answer": labels[i],
        }

        dataset["prompt"].append(input_template_jinja.render(data_input))
        dataset["completion"].append(output_template_jinja.render(data_output))
    dataset = Dataset.from_dict(dataset)
    return dataset

In [6]:
boolq_qa = generate_QA_dataset(contexts, questions, labels)
boolq_qa = boolq_qa.add_column("id", ids).add_column("passage", contexts)

In [7]:
boolq_qa

Dataset({
    features: ['prompt', 'completion', 'id', 'passage'],
    num_rows: 12658
})

In [8]:
print(boolq_qa[0]["prompt"])

<|tasktype|>
boolean question answering
<|context|>
Il persiano (/ˈpɜːrʒən, -ʃən/), conosciuto anche con il suo nome endonimo Farsi (فارسی fārsi (fɒːɾˈsiː) (ascolta)), è una delle lingue iraniche occidentali del ramo indo-iraniano della famiglia delle lingue indoeuropee. È parlato principalmente in Iran, Afghanistan (ufficialmente conosciuto come Dari dal 1958) e Tagikistan (ufficialmente conosciuto come Tajiki dall'era sovietica) e in alcune altre regioni che storicamente erano società persiane e considerate parte del Grande Iran. È scritto in alfabeto persiano, una variante modificata della scrittura araba, che a sua volta si è evoluta dall'alfabeto aramaico.
<|task|>


In [9]:
print(boolq_qa[0]["completion"])


Avendo letto il precedente passaggio, rispondi alla seguente domanda: L'Iran e l'Afghanistan parlano la stessa lingua? Vero o Falso?
<|pipe|>
Risposta: Vero


## 3. Train / dev / test split by passage

`boolq_italian` pairs the same passage with several questions: in train+validation, 1,314 passages have more than one question, and 487 of them are shared between the original train and validation splits. Since the model generates a task **from the passage**, a per-example random split would put test passages in the training set and inflate the scores.

We therefore shuffle the **unique passages** (seed 42) and assign each passage, with all its questions, to exactly one split: 80% train, 10% dev, 10% test.

- **train** — updates the weights;
- **dev** — `eval_loss` during training, early stopping and checkpoint selection;
- **test** — used **once**, for the final metrics in Sections 7–8.

The example ids of each split are saved to `splits.json`.

In [10]:
DEV_FRAC, TEST_FRAC = 0.1, 0.1

passages = sorted(set(contexts))
random.Random(SEED).shuffle(passages)
n_test = round(len(passages) * TEST_FRAC)
n_dev = round(len(passages) * DEV_FRAC)
split_of = {p: "test" for p in passages[:n_test]}
split_of.update({p: "dev" for p in passages[n_test : n_test + n_dev]})
split_of.update({p: "train" for p in passages[n_test + n_dev :]})

data = DatasetDict(
    {
        name: boolq_qa.filter(lambda ex, name=name: split_of[ex["passage"]] == name)
        for name in ("train", "dev", "test")
    }
)

with open("splits.json", "w") as f:
    json.dump({name: list(data[name]["id"]) for name in data}, f)

split_passages = {name: set(data[name]["passage"]) for name in data}
assert not split_passages["train"] & split_passages["dev"]
assert not split_passages["train"] & split_passages["test"]
assert not split_passages["dev"] & split_passages["test"]
data

Filter:   0%|          | 0/12658 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12658 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12658 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'id', 'passage'],
        num_rows: 10154
    })
    dev: Dataset({
        features: ['prompt', 'completion', 'id', 'passage'],
        num_rows: 1245
    })
    test: Dataset({
        features: ['prompt', 'completion', 'id', 'passage'],
        num_rows: 1259
    })
})

## 4. Base model setup

The `sapienzanlp/Minerva-7B-instruct-v1.0` is an instruction-tuned Italian LLM of a size (7B).
 Full fine-tuning of all 7B parameters was not an option with our compute budget.

We load the model in quantized form (bitsandbytes NF4) with `bfloat16` compute (supported by the L40 and the model's native dtype, which avoids fp16 overflows), and add BONITA's special tokens to the tokenizer vocabulary.

Minerva's embedding matrix already has 51,264 rows while its tokenizer uses 51,203 ids, so the four new tokens fit into existing rows and no resize is needed. The original notebook called `resize_token_embeddings(len(tokenizer))` unconditionally, which **shrank** the matrix to the tokenizer size and forced PEFT to save the full embedding and output-head matrices with the adapter (~826 MB). Here the matrix is resized only if it is actually too small.

The rows of the new tokens are not trained (the embedding matrix is frozen): they keep the values already present in Minerva's weights, which are the same at training and inference time.

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained("sapienzanlp/Minerva-7B-instruct-v1.0")
base_model = AutoModelForCausalLM.from_pretrained(
    "sapienzanlp/Minerva-7B-instruct-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

SPECIAL_TOKENS = ["<|tasktype|>", "<|context|>", "<|task|>", "<|pipe|>"]
num_added_toks = tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)
if len(tokenizer) > base_model.get_input_embeddings().num_embeddings:
    base_model.resize_token_embeddings(len(tokenizer))
print(num_added_toks, len(tokenizer), base_model.get_input_embeddings().num_embeddings)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

4 51207 51264


## 5. Training

The model is fine-tuned using **Low-Rank Adaptation (LoRA)** through the PEFT framework, on top of the base model loaded in 4-bit NF4 quantization (QLoRA).

To reduce memory requirements, all original model parameters are frozen and only the LoRA adapter parameters are optimized. One-dimensional parameters, such as normalization weights, are cast to `float32` to improve numerical stability, and the output logits of the head are also cast to `float32` for a more precise loss computation. **Gradient checkpointing** is enabled to further reduce memory consumption during training.

LoRA adapters are applied to **all the linear projections** of the 32 transformer layers (attention `q_proj`, `k_proj`, `v_proj`, `o_proj` and MLP `gate_proj`, `up_proj`, `down_proj`), which the QLoRA paper found to matter more than the rank, with rank $r=16$, $\alpha=32$ (i.e. an effective scaling factor $\alpha/r = 2$), and a dropout probability of $0.05$. This amounts to about 42M trainable parameters, less than 1% of the model.

In [12]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"Trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

for param in base_model.parameters():
    param.requires_grad = False  # freeze the model - train adapters later
    if param.ndim == 1:
        # cast the small parameters (e.g. layernorm) to fp32 for stability
        param.data = param.data.to(torch.float32)

base_model.gradient_checkpointing_enable()  # reduce number of stored activations
base_model.enable_input_require_grads()


class CastOutputToFloat(nn.Sequential):
    def forward(self, x):
        return super().forward(x).to(torch.float32)


base_model.lm_head = CastOutputToFloat(base_model.lm_head)


config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False,
)

model = get_peft_model(base_model, config)

In [14]:
print_trainable_parameters(model)

Trainable params: 41943040 || all params: 3951824896 || trainable%: 1.0613587672483755


Supervised fine-tuning is performed for **at most 4 epochs** using a learning rate of $2\times10^{-4}$ with a cosine schedule and 3% warmup, `bf16` mixed precision, and an effective batch size of 16 (8 per device × 2 gradient-accumulation steps). The loss is computed on the completion only (the generated task), not on the passage.

The model is evaluated on the **dev** split at the end of every epoch; the checkpoint with the lowest dev loss is kept, and training stops early if the dev loss does not improve for 2 consecutive evaluations. The **test** split is never seen during training or model selection. Training is carried out using the TRL `SFTTrainer`, and the selected PEFT model is saved for subsequent evaluation and inference.

In [ ]:
training_args = SFTConfig(
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # float in [0, 1) = ratio of total steps
    bf16=True,
    max_length=1024,
    completion_only_loss=True,
    output_dir="outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    disable_tqdm=False,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,  # type: ignore
    train_dataset=data["train"].select_columns(["prompt", "completion"]),
    eval_dataset=data["dev"].select_columns(["prompt", "completion"]),
    args=training_args,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
model.config.use_cache = False  # type: ignore
trainer.train()
trainer.save_model("Bonita")
tokenizer.save_pretrained("Bonita")

/tmp/ipykernel_403438/3198983120.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/10154 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10154 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1245 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1245 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 51202, 'bos_token_id': 0, 'pad_token_id': 51202}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.260039,0.242644,0.252914,0.924821,1955343.000000
2,0.135831,0.250286,0.189866,0.925023,3910686.000000
3,0.057594,0.255221,0.144539,0.929103,5866029.000000


('Bonita/tokenizer_config.json',
 'Bonita/chat_template.jinja',
 'Bonita/tokenizer.json')

In [16]:
# Training / dev loss curves, for the report.
pd.DataFrame(trainer.state.log_history)

,loss,grad_norm,learning_rate,entropy,mean_token_accuracy,num_tokens,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_mean_token_accuracy,eval_num_tokens,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,1.473474,3.000000,0.000062,1.327745,0.708828,78839.0,0.039370,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.258792,1.726562,0.000127,0.241849,0.924997,157159.0,0.078740,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.203361,1.453125,0.000192,0.191997,0.940004,232835.0,0.118110,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.214596,1.632812,0.000200,0.220714,0.935766,310026.0,0.157480,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.220758,1.773438,0.000200,0.218205,0.934406,387005.0,0.196850,125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.061530,0.664062,0.000036,0.063927,0.974928,5695714.0,2.913386,1850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,0.057662,0.439453,0.000034,0.066634,0.974916,5775619.0,2.952756,1875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,0.057594,0.445312,0.000032,0.065356,0.975156,5851939.0,2.992126,1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,1905,0.255221,132.6276,9.387,1.176,0.144539,0.929103,5866029.0,NaN,NaN,NaN,NaN,NaN


# Part II — Evaluation

Sections 6–8: loading the trained adapter, automatic metrics on the test split, and base-vs-fine-tuned comparison.

## 6. Loading the trained adapter for inference

The tokenizer is loaded from the saved checkpoint, so it contains the same special tokens used in training (the original notebook reloaded Minerva's tokenizer without them, so `<|tasktype|>`, `<|context|>` and `<|task|>` were split into ordinary sub-tokens at inference). The base model is initialized with the same configuration adopted during fine-tuning, the learned **PEFT adapter** is restored from the checkpoint, and the model is set to evaluation mode.

Note that `PeftModel.from_pretrained` injects the adapter **into** `base_model` in place: after this cell, `base_model` is no longer the original model. The base-model baseline in Section 8 is therefore computed with `model.disable_adapter()`.

In [12]:
from peft import PeftModel

# free the training objects, if Part I was run in this session
for name in ["trainer", "model", "base_model"]:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

adapter_path = "Bonita"  # directory written by trainer.save_model above

tokenizer = AutoTokenizer.from_pretrained(adapter_path, padding_side="left")
base_model = AutoModelForCausalLM.from_pretrained(
    "sapienzanlp/Minerva-7B-instruct-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
if len(tokenizer) > base_model.get_input_embeddings().num_embeddings:
    base_model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base_model, adapter_path)

model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(51264, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [13]:
def batch_generate(
    model: transformers.PreTrainedModel,
    tokenizer: transformers.PreTrainedTokenizerBase,
    prompts: list[str],
    batch_size: int,
) -> list[str]:
    """
    Generates one completion per prompt with greedy decoding.

    Minerva's generation_config enables sampling (do_sample=True, temperature=0.4),
    which made the original results change from run to run; here it is disabled.
    """

    predictions = []

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                pad_token_id=tokenizer.eos_token_id,
                max_new_tokens=256,
                use_cache=True,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )

        responses = []

        for inp, out in zip(inputs["input_ids"], outputs):
            responses.append(out[len(inp) :])

        decoded = tokenizer.batch_decode(responses, skip_special_tokens=True)

        predictions += decoded

    return predictions

## 7. Evaluation of the generated tasks: automatic metrics

Three fixes w.r.t. the original evaluation:

1. **One generation per passage.** The test split contains several rows with the same passage (hence the same prompt); with greedy decoding they would all produce the same output. We generate once per unique test passage.
2. **Multiple references.** A passage admits many valid questions, so each generated question is compared with **all** the reference questions of that passage (BLEU and METEOR take multiple references; BERTScore keeps the best-matching one).
3. **Question only.** The instruction is always one of two fixed sentences, every question ends with the fixed suffix *"Vero o Falso?"*, and the answer is `Risposta: Vero/Falso`: including them in BLEU/BERTScore mostly measures whether the template was copied. The text-similarity metrics are computed on the **question** only (instruction, suffix and answer removed); the answer is evaluated separately.

   For reference, keeping the suffix, a question taken from a *different* test passage already scores BLEU 0.27, METEOR 0.40, BERTScore 0.77; with the suffix removed it scores BLEU 0.004, METEOR 0.08, BERTScore 0.69.

The raw generations are saved to `predictions_finetuned.json` and `predictions_base.json` (prompt → generation), so the metrics can be recomputed without re-running the model.

Reported metrics:
- **format adherence** — fraction of outputs with both a question and a parseable `Risposta: Vero/Falso`;
- **BLEU / METEOR / BERTScore-F1** — generated question vs. reference questions;
- **Vero rate** — fraction of generated answers that are `Vero` (62% in the references: a generator that always answers `Vero` is not detected by the metrics above);
- **question match rate** and **answer accuracy on matched questions** — when the generated question coincides (after normalization) with one of the reference questions, its answer can be checked against the reference label. For all other questions, answer correctness cannot be checked automatically against the benchmark and needs a judge (human or model).

In [14]:
LABEL_RE = re.compile(r"Risposta:\s*(Vero|Falso)", re.IGNORECASE)
QUESTION_SUFFIX = "Vero o Falso?"


def parse_task(text: str) -> tuple[str, str | None]:
    """Splits a generated or reference completion into (question, label).

    The instruction and the fixed " Vero o Falso?" suffix are removed, so that the
    text-similarity metrics are computed on the question itself.
    """
    text = text.replace("<|pipe|>", "")
    match = LABEL_RE.search(text)
    label = match.group(1).capitalize() if match else None
    body = text[: match.start()] if match else text
    for variant in INSTRUCTION_VARIANTS:
        body = body.replace(variant, "")
    body = body.strip().removesuffix(QUESTION_SUFFIX)
    return body.strip(), label


def normalize(question: str) -> str:
    return " ".join(question.lower().replace("“", "").replace("”", "").split())


# group the test rows by passage: one prompt, several reference (question, label) pairs
test_refs = defaultdict(list)
for prompt, completion in zip(data["test"]["prompt"], data["test"]["completion"]):
    test_refs[prompt].append(parse_task(completion))

test_prompts = list(test_refs)
print(f"{len(data['test'])} test rows -> {len(test_prompts)} unique passages")

1259 test rows -> 1034 unique passages


In [ ]:
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")


def compute_metrics(predictions: list[str]) -> dict[str, float]:
    parsed = [parse_task(p) for p in predictions]
    gen_questions = [q for q, _ in parsed]
    gen_labels = [label for _, label in parsed]
    ref_questions = [[q for q, _ in test_refs[p]] for p in test_prompts]

    matched, correct = 0, 0
    for (question, label), prompt in zip(parsed, test_prompts):
        ref_labels = {normalize(q): lab for q, lab in test_refs[prompt]}
        if normalize(question) in ref_labels:
            matched += 1
            correct += label == ref_labels[normalize(question)]

    bert_f1 = bertscore.compute(predictions=gen_questions, references=ref_questions, lang="it")["f1"]
    labelled = [label for label in gen_labels if label is not None]

    return {
        "bleu": bleu.compute(predictions=gen_questions, references=ref_questions)["bleu"],
        "meteor": meteor.compute(predictions=gen_questions, references=ref_questions)["meteor"],
        "bertscore_f1": sum(bert_f1) / len(bert_f1),
        "vero_rate": labelled.count("Vero") / len(labelled) if labelled else float("nan"),
        "question_match_rate": matched / len(parsed),
        "answer_acc_on_matched": correct / matched if matched else float("nan"),
    }

In [16]:
predictions = batch_generate(model, tokenizer, test_prompts, 32)

# save the raw generations, so the metrics can be recomputed without re-generating
with open("predictions_finetuned.json", "w") as f:
    json.dump(dict(zip(test_prompts, predictions)), f, ensure_ascii=False, indent=1)

for prompt, prediction in list(zip(test_prompts, predictions))[:5]:
    print("GENERATED:", parse_task(prediction))
    print("REFERENCES:", test_refs[prompt])
    print()

  0%|          | 0/33 [00:00<?, ?it/s]

GENERATED: ('“La persona soccorritrice può essere citata in giudizio?”', 'Falso')
REFERENCES: [('“Le leggi del buon samaritano proteggono chi aiuta in caso di incidente?”', 'Vero')]

GENERATED: ('Lo zucchero a velo è uguale allo zucchero normale?', 'Falso')
REFERENCES: [('Lo zucchero da cucina è uguale allo zucchero a velo?', 'Vero')]

GENERATED: ('Il ghiottone è la stessa cosa di un tasso?', 'Falso')
REFERENCES: [('Il ghiottone è la stessa cosa di un tasso?', 'Falso')]

GENERATED: ('Ci sarà un altro film di Now You See Me?', 'Vero')
REFERENCES: [('Ci sarà un terzo capitolo di Now You See Me?', 'Vero'), ('Ci sarà un Now You See Me 3?', 'Vero')]

GENERATED: ('Ci sarà una nuova stagione di Cut Throat Kitchen?', 'Falso')
REFERENCES: [('Ci sarà una nuova stagione di Cutthroat Kitchen?', 'Falso'), ('Ci saranno nuovi episodi di Cutthroat Kitchen?', 'Falso')]



In [20]:
finetuned_metrics = compute_metrics(predictions)
finetuned_metrics

{'format_adherence': 1.0,
 'bleu': 0.2764822425391711,
 'meteor': np.float64(0.45054105005434236),
 'bertscore_f1': 0.8319151086779581,
 'vero_rate': 0.5406189555125726,
 'question_match_rate': 0.0735009671179884,
 'answer_acc_on_matched': 0.8289473684210527}

## 8. Base vs. fine-tuned comparison

The baseline is Minerva-7B-instruct **without** the LoRA adapter, on exactly the same prompts and with the same greedy decoding. It is obtained with `model.disable_adapter()`: in the original notebook `base_model` was used after `PeftModel.from_pretrained` had injected the adapter into it, so the "base" numbers were in fact a second run of the fine-tuned model.

In [18]:
with model.disable_adapter():
    predictions_base = batch_generate(model, tokenizer, test_prompts, 32)

with open("predictions_base.json", "w") as f:
    json.dump(dict(zip(test_prompts, predictions_base)), f, ensure_ascii=False, indent=1)

for prediction in predictions_base[:5]:
    print(repr(prediction))

  0%|          | 0/33 [00:00<?, ?it/s]

'\nLa legge del buon samaritano protegge anche i soccorritori da danni?\nIn alcune giurisdizioni, come il Regno Unito, gli Stati Uniti e l\'Australia, esiste una legge chiamata "Duty of Care" o "Good Samaritan Law", che può fornire protezione legale ai soccorritori che hanno assistito una persona ferita, malata, in pericolo o incapace di provvedere a se stessa. Questa legge fornisce protezione solo quando la persona assistita era in una situazione di emergenza o quando la persona che assisteva aveva ragionevoli motivi per credere che la persona assistita fosse in una situazione di emergenza. In altre parole, la legge non fornisce protezione se la persona che assisteva credeva erroneamente che la persona assistita stesse bene o se la persona che assisteva stava semplicemente cercando di fare qualcosa di buono senza alcuna intenzione di ferire o causare danno.\nLa legge del dovere di cura è stata introdotta negli Stati Uniti dal California Labor Code (California Labor Code § 3312) e dall

In [19]:
base_metrics = compute_metrics(predictions_base)

pd.DataFrame({"base": base_metrics, "fine-tuned": finetuned_metrics})

,base,fine-tuned
format_adherence,0.000000,1.000000
bleu,0.004069,0.276482
meteor,0.106293,0.450541
bertscore_f1,0.584038,0.831915
vero_rate,NaN,0.540619
question_match_rate,0.000000,0.073501
answer_acc_on_matched,NaN,0.828947
